# Instrument LAN Connection Test
### Rigol DG2102 (Waveform Generator) + Keysight DSO-X 6004A (Oscilloscope)

Both devices connect via **LAN / VXI-11**.  
VISA resource string format: `TCPIP0::<IP>::INSTR`

**Edit the IP addresses in the cell below before running.**

In [2]:
# ── User configuration ──────────────────────────────────────────────────────
DG2102_IP   = "192.168.68.33"   # Rigol DG2102 LAN IP
SCOPE_IP    = "192.168.68.52"   # Keysight DSO-X 6004A LAN IP
# ────────────────────────────────────────────────────────────────────────────

DG2102_VISA  = f"TCPIP0::{DG2102_IP}::INSTR"
SCOPE_VISA   = f"TCPIP0::{SCOPE_IP}::INSTR"

print("DG2102 resource :", DG2102_VISA)
print("Scope  resource :", SCOPE_VISA)

DG2102 resource : TCPIP0::192.168.68.33::INSTR
Scope  resource : TCPIP0::192.168.68.52::INSTR


---
## 0 — List all visible VISA resources
Run this first to confirm pyvisa finds devices on the network.

In [3]:
import pyvisa

rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Visible VISA resources:")
for r in resources:
    print(" ", r)

ValueError: Could not locate a VISA implementation. Install either the IVI binary or pyvisa-py.

---
## 1 — Imports & sys.path setup

In [5]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Make sure the repo root is on the path so `from instruments import ...` works.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..") if os.path.basename(os.getcwd()) != "AOM" else os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo root:", REPO_ROOT)

from instruments.Rigol       import DG2102
from instruments.Oscilloscope import DSOX6004A

print("Imports OK")

Matplotlib is building the font cache; this may take a moment.


Repo root: /Users/ball/Github/AOM


/Users/ball/Github/AOM/instruments/Bristol.py:23: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  if self.pathToLib is 0:
/Users/ball/Github/AOM/instruments/Bristol.py:60: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  if self.h is -1:


NotImplementedError: Location of niDAQmx library and include file unknown on darwin - if you find out, please let the PyDAQmx project know

---
## 2 — Connect to instruments

In [4]:
awg   = DG2102(DG2102_VISA)
scope = DSOX6004A(SCOPE_VISA, timeout_ms=30000)

print("DG2102  connected:", awg.connected)
print("Scope   connected:", scope.connected)

NameError: name 'DG2102' is not defined

---
## 3 — DG2102: *IDN? and basic checks

In [ ]:
# Identity string
idn_awg = awg.gpib_query("*IDN?")
print("DG2102 IDN:", idn_awg)

# Read current CH1 frequency and amplitude
print("CH1 freq (query)  :", awg.gpib_query("SOUR1:FREQ?"))
print("CH1 volt (query)  :", awg.gpib_query("SOUR1:VOLT?"))
print("Errors            :", awg.get_error())

### 3a — DG2102: output a 1 kHz sine on CH1

In [ ]:
awg.set_func(1, "SIN")
awg.set_freq(1, 1e3)        # 1 kHz
awg.set_amplitude(1, 0.2)   # 200 mVpp
awg.set_dc(1, 0.0)          # 0 V offset
awg.set_output(1)           # output ON for both channels

print("CH1 function:", awg.gpib_query("SOUR1:FUNC?"))
print("CH1 freq    :", awg.gpib_query("SOUR1:FREQ?"))
print("CH1 volt    :", awg.gpib_query("SOUR1:VOLT?"))
print("Errors      :", awg.get_error())

### 3b — DG2102: output a TTL-like square on CH2 (scope trigger source)

In [ ]:
awg.set_func(2, "SQU")
awg.set_freq(2, 1e3)         # 1 kHz square
awg.set_amplitude_wfm(2, 0.0, 3.3)  # 0 → 3.3 V TTL-like swing
awg.gpib_write("OUTP2 1")   # CH2 output ON

print("CH2 function:", awg.gpib_query("SOUR2:FUNC?"))
print("Errors      :", awg.get_error())

---
## 4 — Oscilloscope: *IDN? and basic checks

In [ ]:
print("Scope IDN:", scope.idn())
print("Errors   :", scope.get_error())

### 4a — Scope: configure for triggered acquisition
CH1 = signal (from DG2102 CH1), CH2 = trigger (from DG2102 CH2)

In [ ]:
scope.setup_qds_apd_triggered_acquisition(
    signal_channel   = 1,
    trigger_channel  = 2,
    time_range       = 2e-3,   # 2 ms window (one full 1 kHz period visible)
    time_position    = 0.0,
    trigger_level    = 1.65,   # ~midpoint of 3.3 V square
    signal_scale     = 0.1,    # 100 mV/div for 200 mVpp sine
    signal_offset    = 0.0,
    trigger_scale    = 1.0,
    trigger_offset   = 1.65,
    signal_impedance = "ONEMeg",
    trigger_impedance= "ONEMeg",
    points           = 1000,
    acquire_type     = "NORMal",
)

print("Setup complete")
print("Time range :", scope.ask(":TIMebase:RANGe?"))
print("Trig level :", scope.ask(":TRIGger:EDGE:LEVel?"))

### 4b — Scope: capture a waveform and plot

In [ ]:
t, v = scope.read_waveform_ascii(source_channel=1, points=1000, digitize_first=True)

print(f"Points received : {len(t)}")
print(f"Time range      : {t[0]*1e3:.3f} ms  →  {t[-1]*1e3:.3f} ms")
print(f"Voltage range   : {v.min()*1e3:.1f} mV  →  {v.max()*1e3:.1f} mV")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t * 1e3, v * 1e3, lw=1)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Voltage (mV)")
ax.set_title("CH1 waveform — 1 kHz sine from DG2102")
ax.grid(True)
plt.tight_layout()
plt.show()

### 4c — Scope: quick measurements

In [ ]:
vpp  = scope.measure_vpp(channel=1)
vavg = scope.measure_vavg(channel=1)
freq = scope.measure_frequency(channel=2)

print(f"CH1 Vpp  : {vpp*1e3:.2f} mV")
print(f"CH1 Vavg : {vavg*1e3:.2f} mV")
print(f"CH2 freq : {freq:.1f} Hz")

---
## 5 — End-to-end round-trip test
Set DG2102 to several frequencies, measure each one with the scope.

In [ ]:
import time

test_freqs = [500, 1000, 2000, 5000]   # Hz
results = []

for f_set in test_freqs:
    awg.set_freq(1, f_set)
    awg.set_freq(2, f_set)
    time.sleep(0.3)   # let scope re-trigger

    # Adjust timebase to show ~2 periods
    scope.set_timebase(time_range=2.5 / f_set)
    time.sleep(0.1)

    f_meas = scope.measure_frequency(channel=2)
    vpp    = scope.measure_vpp(channel=1)
    results.append((f_set, f_meas, vpp))
    print(f"Set {f_set:5d} Hz  →  measured {f_meas:8.2f} Hz,  Vpp = {vpp*1e3:.1f} mV")

# Simple pass/fail: measured frequency within 2 % of set value
print()
all_pass = True
for f_set, f_meas, _ in results:
    err_pct = abs(f_meas - f_set) / f_set * 100
    status = "PASS" if err_pct < 2.0 else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  {f_set:5d} Hz  error={err_pct:.2f}%  [{status}]")

print()
print("Overall:", "ALL PASS" if all_pass else "SOME FAILURES")

---
## 6 — Teardown

In [ ]:
awg.set_output(0)    # DG2102 outputs OFF
scope.run()          # return scope to free-run
print("Done. Instruments released.")